In [8]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader
import copy

In [9]:
num_classes = 4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batchsize = 16

In [10]:
class EEGDataset(Dataset):
    def __init__(self, data_path, labels_path):
        self.data = np.load(data_path)
        self.labels = np.load(labels_path)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get the EEG data and corresponding label
        eeg = torch.tensor(self.data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return eeg, label

# Example usage
train_dataset = EEGDataset("eeg_dataset/train_epochs.npy", "eeg_dataset/train_labels.npy")
val_dataset = EEGDataset("eeg_dataset/val_epochs.npy", "eeg_dataset/val_labels.npy")
test_dataset = EEGDataset("eeg_dataset/test_epochs.npy", "eeg_dataset/test_labels.npy")

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batchsize, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batchsize, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batchsize, shuffle=False)

print(len(train_loader))
# Check one batch
for eeg_batch, label_batch in train_loader:
    print("EEG Batch Shape:", eeg_batch.shape)  # (batch_size, 14, 640)
    n_channels = eeg_batch.shape[1]
    samples_per_data = eeg_batch.shape[2]
    print("Label Batch Shape:", label_batch.shape)  # (batch_size,)
    break


79
EEG Batch Shape: torch.Size([16, 8, 1088])
Label Batch Shape: torch.Size([16])


In [11]:
sampling_rate = 256
print(n_channels)
print(samples_per_data)

8
1088


In [12]:
# classifier
class EEGNet(nn.Module):
    def __init__(self, n_class, n_channels, total_samples, sampling_rate, F1 = 8, F2 = 16, D = 2):
        super(EEGNet, self).__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=F1, kernel_size=(1, sampling_rate//2),
                      padding="same", bias=False),
            nn.BatchNorm2d(num_features=F1),
            nn.Conv2d(in_channels=F1, groups=F1, out_channels=D*F1, kernel_size=(n_channels, 1), bias=False),
            nn.BatchNorm2d(num_features=D*F1),
            nn.ELU(),
            nn.AvgPool2d(kernel_size = (1, 4)),
            nn.Dropout(0.25)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=D*F1, groups=D*F1, out_channels=D*F1, kernel_size=(1, sampling_rate//8), 
                      padding="same", bias=False),
            nn.Conv2d(in_channels=D*F1, out_channels=F2, kernel_size=(1, 1), groups=1, bias = False),
            nn.BatchNorm2d(num_features=F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size = (1, 8)),
            nn.Dropout(0.25),
        )

        self.block3 = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features = F2*(total_samples//32), out_features=n_class)
        )


    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        return x

In [13]:
model = EEGNet(12, n_channels, samples_per_data, sampling_rate).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)   # optimize all parameters
criterion = nn.CrossEntropyLoss()

best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

for epoch in range(20):
    
    # Train the data
    model.train()
    running_loss = 0.0
    for inputs, label in train_loader:
        # Inputs are in shape (batch_size, channels, timepoints)
        # Need to transform to (batch_size, 1, channels, timepoints)
        inputs = inputs.unsqueeze(1).to(device)
        label = label.to(device)
        
        model_output = model(inputs)
        loss = criterion(model_output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    # Validate Data
    model.eval()
    val_acc = 0
    with torch.no_grad():
        for inputs, label in val_loader:
            # Inputs are in shape (batch_size, channels, timepoints)
            # Need to transform to (batch_size, 1, channels, timepoints)
            inputs = inputs.unsqueeze(1).to(device)
            label = label.to(device)
            
            model_output = model(inputs)
            pred = model_output.argmax(dim=1, keepdim=True)
            val_acc += pred.eq(label.view_as(pred)).sum().item()

    # Print to see status
    print(f'Epoch {epoch+1}, Training Loss: {running_loss/len(train_loader)}, Validation Acc: {val_acc/len(val_loader.dataset)}%')

    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

Epoch 1, Training Loss: 2.3170792633974098, Validation Acc: 0.5682656826568265%
Epoch 2, Training Loss: 1.5445056663283818, Validation Acc: 0.7380073800738007%
Epoch 3, Training Loss: 0.9755868281744704, Validation Acc: 0.8856088560885609%
Epoch 4, Training Loss: 0.6332542022949532, Validation Acc: 0.933579335793358%
Epoch 5, Training Loss: 0.42526970030386235, Validation Acc: 0.9520295202952029%
Epoch 6, Training Loss: 0.32455363935684856, Validation Acc: 0.966789667896679%
Epoch 7, Training Loss: 0.25281197785199444, Validation Acc: 0.974169741697417%
Epoch 8, Training Loss: 0.23102021104172815, Validation Acc: 0.977859778597786%
Epoch 9, Training Loss: 0.1989157403807474, Validation Acc: 0.9704797047970479%
Epoch 10, Training Loss: 0.17658552235063119, Validation Acc: 0.985239852398524%
Epoch 11, Training Loss: 0.15598876781384402, Validation Acc: 0.9704797047970479%
Epoch 12, Training Loss: 0.15298922532061232, Validation Acc: 0.974169741697417%
Epoch 13, Training Loss: 0.126978143

In [14]:
model.load_state_dict(best_model_wts)
# test
with torch.no_grad():
    n_correct = 0
    n_samples = 0
    for eeg, labels in test_loader:
        eeg = eeg.unsqueeze(1).to(device)
        labels = labels.to(device)
        outputs = model(eeg)
        
        _, predictions = torch.max(outputs, 1)
        n_samples += labels.shape[0]
        n_correct += (predictions==labels).sum().item()
        
    acc = 100 * (n_correct/n_samples)
    print(f'accuracy = {acc}')

accuracy = 97.4074074074074
